In [ ]:
print("[cell] 01 run parameters (kapi --env): environment variables of this r", flush=True)
# run parameters (kapi --env): environment variables of this run
import os
os.environ['FVB_ARMS'] = 'B1'

# FLAG 2027 - NB-189: FVBIND-01, ImageBind audio tower re-trained the ImageBind way (big batch, identity face anchor), GPU

**What (Experiment/FVBIND-01/README.md, registered 14/11 before any number).** LoRA C1 recipe on the ImageBind-huge audio tower,
trained only on the auxiliary VoxCeleb2 + VoxBlink persons of the RFS pool (clean_ext / lid_strict, BN42-clean audio cache):
B0 = LoRA C1 recipe (64 persons), B1 = 2 048 persons per step (GradCache), B2 = B1 + anchor [C5s | FaRL] + linear voice head,
IB0 = frozen ImageBind (reference). Standalone stream on the SAFE-04 fold tables; gate 1 is read locally (`run.py gate`).
No dev, no CB, no Evaluation, no MAV / v4 in training.

In [ ]:
print("[boot] FLAG_189_fvbind | python", __import__("sys").version.split()[0], "| start", __import__("time").strftime("%Y-%m-%d %H:%M:%S"), flush=True)  # heartbeat rule: first output, before any import
print("[cell] setup: files + deps", flush=True)
import os, sys, json, gzip, base64, subprocess, shutil, time
from pathlib import Path
MODE = os.environ.get("RFS_MODE", "LAB").upper(); SMOKE = os.environ.get("FLAG_SMOKE") == "1"
W = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./rfs_work"); W.mkdir(exist_ok=True)
CODE = W / "rfs"
for rel, blob in (("train/assemble.py", "H4sIAAAAAAAC/61ZaW/bSBL9rl/R4H4w6ci0ZXuTjBwFmBweDJDNDDJZIYAgEC2xKXElHuBlOo73t++r6uZly0kwWCMx+6iurqquel3Vtizr0/VfJ2eTqdhsVSp8WYpMiusPv/7mTZ6fC1uKU7HC/7UjnjXDZ8K+2VYXYjAXqUKKYivjrViVsb9XYl1KkQW5V2QyjN30VtiqLtw4/ToW1SV9HXck8JPeFtskFjLPVbTaK6I8OUnKQrz7/RNaxEKKd6a1altrbt1szSQaq6ahp7AbC8UL5E11Kd6N3iQQa81qVjIR9qf3v77713tx9KZM1mJy5EyFnIpArpX3KgvW+zD9lgXhKsBvuVHBa+9Vnu7D4jWkv4UWSagJcxmBIjs/n+BTV7ym0muqwRphc1scsU2OxEzAYBvBPe99vNmH+dYt6sIZjyC996qaeOskSveqUN+q87YNniTjNxYAnQhMDP9iW4qiFLT62CP13XVeOVf6nKbiVZXUWFafr75Vq9end8RmzGzu+2yKrUq0GRoWWK7VbQZGNou+w9FBgS3Zb18mhfATaPVRfrwS2yTFts/gHq4gf5kaDtQeGpJ1zbP1a6+j0DM8ejocdS3LGoVRmmSFkNkmlVmuxiJMRkGWRCKVxXYfroQh+BPdhjguI+gmcxGnY9DFPpr4l/qj0fXFOcTey2jlS/FlCgpX5jLL5K39ZUy9YJ/I4uLcGY1Gvgrg7dlG2cegyJ0pezGEIoXXUtthqk8WhuA4YNMUGVnJp18IlFuxSkI9kcMdtTnJMYtQxWJblsSBI2rjsMq0C4XFTEBam7ZenC0dd52kt7bDs0GSIShDxBJNTqZLLRr9hAHN5OJjEqupWCdxEcalaqdX0gff/0LRMA/COCyUja0cV+739gTug84CNMtmc4d7vDxTRZnFRGFsg/W+nSVJQUYujHkyrMxxCMq36Uh43nGzzT5Z2USFPQgAcErZWARWsQ1VKe4wcw+zkYvd0Yp766rZL4PyZkMcTbNhpvZmwxQbdjvBfTHVlxeq8rrUIdukrqrDvMhtR6h9rthMhrssa6+QAKYcardnPU/qt2qvVoBIhBOZhbDx4gX8ncOrGfnnhQOkESB/sw/jHahXAJkIJx9vSqBehOOnkLoSkSyIchOSthl8YNOe+vVYzMHwbjcViyWf8o5O2crTnahCXyBIBIGVAFRZLkeW7dyPv7+gLMjxIkGwJeqKQ7Rb3PpTTgttC0pZY2GxbtxYWU7nXcRvxoNkzFzMTIdtyWtb0iACZeq7mZK+Bxyx2V2ky+BOR3+X358OoMdyxsIvblM1y4sMflL9DIchWA1ZtLJsupj3c3aeqXYm0+soyRKr/W4s4AnAg4iNYlv6joA9cEIWN7ERDdNB6FFqOT1T0c8X7GsQZFEz75oZbg7a4Q4b3xPwEfOGZP0UCbtzTaEew7XIj5fOYHNo4OaqgGvLcl/YlmfBZ+NoDE9BvKepgjG/HNC8pzTchnyAPkZfciIa4q8Zqysawe+H6s9/uP9PGUcf8fet84jmkHk6ZW+2OJlHu1JqMWTYXEXNnpRzPEVxYMfRE5bgEDxkiJstc4F4HFF0H5VAZnuvYruKIMTk/OWZwxdVLOPBhdUpd72g+LeW4hm8HmBnBxHCfYeIwgxQQc8soMXu/vSuurc0cCDPIUt8DVOzAIpGLuhV4ix5LaLfrEVrKY4FiRVE3c7zhztXzc7zn9y5MjtXvZ3nT+wMg/AkAG64JQac/hVAsNrc57J6CPIGdP1wXSBr280WyzFhJ38p1LmB6MaX0OUQGeUi3ECk8JfCgxt1xR86cFrfom1l0JaDqTrvY+xTwGnSXHI/yqNgvjZP9B7g6HfA83tcHmJpd7Ag7SB0B05jjRV+3gPSAxzviPa+H5cHvZQ8AssQVVZ3F2knXD722kOU7CzL74KZQfAD8N1gN3aKo2UTkpT/QCHbIuNaRmENO9JxDjv9QfGqRpH5jxRpvF7Tklf1iU/vUkOcGmKiWP6fERw7HzIC+8Z3rHAD/xgQWgbkGDSljlQeGvC+0XjXR82/C32Pg7261LHez+denHFCFmJSl2M6STtcnsHjOXHjnMumxO2ZztsowEKfU1291KZ6wBRKaMVb5PdYfMTHedQNdhl+wSF1KL2hEhb2eyQNnVau0pklYNYtVqlsRgbT6t/Iaiz+k25Ag+RvRvwX57B1TumQTenQmMcuDoxdDsaY3SaSaYOKexfDAGfHZI3W2HIW03OdcO7J1xIcKETf4FhVRsqg1ovXiR/GmxnwOTh5afFBt5y411+gT/1O56M0jM2pOOPSbDEhchIJKwC/Dieekck7z7oAojhj1wjj4qVW5B/izee35mhxCbu4Zabi7V9zkXSDp4GScB6V60I515Uy7vixqXvXpS5+RYRHhcducvIaxVxEl5Io4EwXqA5RIW4lC1CmHXTC22KKEpuqmi6UF1zAmKPHmRwP+gjzTMVFvlxML/WijBeZMis11VXsOEsyEx0qu4UA5dCn4sYkpD+rwUUrGWid0GNAJN7+8Vm8/fcfv5P5qS7VwWJfnv3yArXV5BfolhRXzMBXFZQMBeae89y5Q5ZYU32jUP7CZAiPNVV4vC2FZObbiF2/d93VD6KgTKHJ0L/dIvG4prd7Qd/WkbWbb2WqFpMlox6uJh9hOiFow1Zm1mnryXoxHYupqWnrgK5oSAC5hgHHV2rO9+CYNQS/x0TaTwwVGaCVimCrDthRuVm1TX72EZjUkpEATauZ1sf04cCVO+WyvKlpewUR3+j969Y8x+nU1EAEaWrWEChznde+1PTomVrCNpJWfGivQEs/jxEQfeghvX4AswYwzMAReJQp4X93ghp0Ao9Spy6+Ayu7C83lFtIxZoBSZbf2WPIajRYz+lCXsy4g3gPmgxKEfqqnpKj+hhTVUIpKSwHw/ZEUgcfJ5MCYTSoSeJRfDqY4JaGZOpiRl0IJ5JZDq5tbvfI43RzONdd75SEFratDVuGktO9ON5XDW+I0ZzKgODmjJFf3K91vH8bgLLYJY74pmhc699dsU0bAqz+pl/VerHY658UDkhHfk01j1TTWOnPQM1SPmS+Pm9yVaQnQqNHdIBY96uIsfd+TRgTbOjmhqnNnPJrETF0Wk2jwCnRlXtoM4NJbGA+50c4PM9sg7+xzVip6vkSB4SU77mqW1wiiOfPtvx4hZ0WdO6cUfFBwXDE4tlFd5/wAiUc6DKsY/+0613zfN3evDqHWR69lm21iD9N+HFBEZlJNJuM2k6F+esBNl1ZiYUFUq6mtOt4PXdn4MSTm5dxr99Gd1qNbIvLmhobbD7lWQzXnPTXnPTUfROy8p+a8p2Y1VHP+pJrzJ9WsPFSRQxamzgSLs4MMGjDoAMXq6BqFHCOqTu4PmAFRTmajBdTpTEDtLtpbGg70hkh3mrBvaeqqpaAmz3P0txQ6Mzc0uuM0OWFz6U6em3sEA7ms1Fd6OsbFb5k/9yAcj4/pKdKuDiyl1MQggHYhil52Ey6BPVOpaPV0iwoSneLh9uy/FLx38XAdIXzvDwuk/+zE8vRqAE2bZiEBg/7b1dTSj6fmBn5iE7jzvsy3JuxHI2jiebGMlOdxIup5hIaeZ2k41NA4+h9GRZ9QcxsAAA=="), ("train/gender.csv", ""), ("train/rfs_train.py", "..."), ("ref/flag_final_core_092.py", "..."), ("ref/manifest_final.json", "H4sIAAAAAAAC/71X247byBF9D5B/KOhpBitp+sbupgcJYA9sZAEnG3sTB0FgCM2+aGhTpEBSmrEX/vecpih5bGSDBAjyMBqy2F116tSlq3/57W9oMXwaxrhbPKPF21er12LFSkFXv6P5xVr6gd68ekb72K/2ru6p6bbUdv1uoC7ReB9p33cfoh9joOR8pBs6djX+x10VQ6jb7UBVTF0f6bWYNrqmHtxYd+2S2oTl7XFJbqB0GGJPw9hHtxuun9Gbd6u/3dcD7NIoAeLHndvGF3Ubbl53b5+v7rinuoIcOvBzpKsfCAuobmFks41twMZUN3G4vqWnr/T6LXUtXY1yCQUZA/6O18vvt9Hd89Wrv/788hYfaPDZge/Rd+S7+64fp8cxDiPg49sw1n5YLCdyT+5stuD3HxDQYpTTF1rU1fzQpvPDcYH/77/Z2P6nO08PYOCrjlFuplBs4uPYu7OeX/IPLaqm8x9z1B/uj7NiWoQ6JwIXls2CXRdik1d1+9i6+ubhFJFV4/ptXB0lxdZjSb+kXXQtdcccLeRE6t0uXjIk1T24kYyGJb50n2ObQdKXC9I5hboepiZ8M5KCi5Nfoe/2mwQJW9snkuMkKU6Spp/eGGOz4CGcBfwkQKLvB8gkRPMS1+8O++zzWQLvhq4dnhof3eGkaNY7xBiGM5100k2ntSTz73v8fJkcgxuxHeqTwmm9YqWelvJSXPy/T5s+HuvzwhMFv0J5jgbTSUiZItPRcCaFcN6VXHpeeSVU6VgwPPnIz+nhxti7h5tjPa6QISvfuGGoUx0zYwtVFc57qYpQ6lJ4mZJKeBEFKyR3KUgng2PxnGvuEGLsUdc3D+4ojtGLGVrfVYdhXAm1Siczp2rKJioXhJEmFaLk2lZeBpl8KE0MyVWuLKz0JkgdZhOz576p96uM+aR/70Z/z1XWJ0UVtBLWWqaC1jGmwLyvouCFc/gQkuHeK764BCIXxpnYKam/BvDf0EPxcX9qbbm1XOX+dn1J4blE/ls+JlWPOeHoairP62+K4v0csm7MgBdPmtLUH7u2+XRLFUpqzLDQyHKBHRVdvWy3aE3314Rar1vgob57GL4yMHSHHr3A9WMNP8bNzrV1Qs+60IIusW73n7NVk7QzPlbIAc+RW1VARrAyMO2sLcB0aUobKimNiMEZWWhfJRGMKEVMJT/n3QR5xr9perX+0FUNuhcsWBWViQopW0qkGNcVc1brElmQVIzIQhOF4dFp7YR1OgWrXYiWSasY9+obC8OD22/ePNWuk0dKGM6rCJRamFKoonQK+EqppEiQWJOM80xqzg3zKqbkC6cZUNhZe8M3E5lnWsCJL2Lk0kiNtE2VTyYpUVXeCKG5rQQ3JhgmiwJJLSuu8ciSNfAv8VSc1Xa929zx9T5zv4joVxzlFlEPupTMCR0EADpbqcI6sGyDtsGzytkiJC2C9FrZwgmhIjd+VnppoesP6F9ZsUiImNKu1IFjU4Euoa012ju4GgtUouasjIVFKWpTeNRktLJAGVqukkjfKwbDJ8TGB88NIGrGiwJYUhmEEkk6hyyIOtqqMJGhGI3hqtAc7EvhhPeJiXRB3B/aje9aMOzHC2jO4ZpEuqGSmZCuQG9QTgVfSWbR0UrlHPNaF1alAsYs+kcsTSVTaQtEYNY9hS1rx4HcjsMlgMhYI3xE2iZnjPQSCeeSLCuFwOoqSleWxrlShdJpWcRgkk3MFE5a5kv9tZqmEeFSOqluXbPZO5x6F1s8OCuN46r0YEuhTry1DnRhwLKFcSDKs1BaB+qFCFUpWBAgk8GfSlj5f0mW/3VMh0O1q4d8juFAS7HHeBDXn+v9lI+5tlAGQUYUBzMGr4gqQ5FHhVaueYgom4rHqFG/JUdJ+WSZFgVSIlVfyQ8RLbqP+XhfvHxE8wb7tO+65hm96x7vYoPORQWBH4yt3b5B6z60XUq1r7HwD69oV/d919Of+26LkN11u/0BWm6O3aPPm9GR//j83WpSREd+cxQk1KxqmU28aOr2IxkyXJ4t3BAvSAlN+dQafsUg5qHz7iXd3dGLv6/+dLf6+fmS+jhE1/t7QkPL4+QIScAk2dcVoIXrNf0F8yC9iO0WEyj1h+zUaVahJh5jcz35/hp9/+A4M5iJ5+l5NWBmbZ68X2Y3bMXO/tM0mdEDRuvu4ZaK8gmJs3Opcdvt6bjpcN70dKxD7ODl+ZZQXAPvDuNfyGfZjiose3o5gO85PuTaMInzCE/BjY6upkdM88vLlcNgHCsxhYqZy5scS1HOYEDFJQJ99PU+kr/vhtiSS+M8fs6aNIJWN+Dyp59e3V6EmvY434G0Hgdo2IJlJGqgLcYAumpxJOIFO2jF1gqXBHPD2ZL2JSPJ16Wk1e/zg1bXt08uSfVADreB3b6pHXIeeqcWlCMOi6eIOmrjAw1g1ucLxJouFy7cdl7h9oSe8q9RjQ/5RjUf9tys5sBnTFdQ/yxDFWpJFlBvCXhOkhk8gJ6M5jlm4ijbuOuCQzoh5fx99B/pitu15nQcCA9lgRo4l9kalffln1quD3wsDgAA")):
    p = CODE / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_bytes(gzip.decompress(base64.b64decode(blob)))
TRAIN = CODE / "train" / "rfs_train.py"
def run(args, env=None):
    """chay rfs_train.py, chuyen stdout tung dong ra log"""
    p = subprocess.Popen([sys.executable, "-u", str(TRAIN), *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, **(env or {})})
    for line in p.stdout: print(line, end="", flush=True)
    assert p.wait() == 0, "rfs_train.py failed"
if MODE == "L" and not SMOKE:
    for c in ("pip -q install iopath ftfy regex fvcore", "pip -q install --no-deps git+https://github.com/facebookresearch/ImageBind"):
        print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout[-300:], flush=True)
import torch
print("mode", MODE, "smoke", SMOKE, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count(), flush=True)

In [ ]:
print("[cell] inputs: bundle, lid, roots", flush=True)
IN = Path("/kaggle/input")
def find_all(name):
    return sorted(p for p in IN.rglob(name)) if IN.exists() else []
WORK = W / "work"; WORK.mkdir(exist_ok=True); ART = W / "rfs_art"; BUNDLE = None; LID = ""
if MODE == "B":      # ghep FLAG_162 (a, b, c) + FLAG_160 (whv3 a, b, c) thanh bundle (CPU, vai phut)
    def dsdir(slug_part):
        r = [p for p in IN.rglob("*") if p.is_dir() and p.name.startswith("flag2027-feats-") and slug_part in p.name]
        assert r, f"thieu dataset *{slug_part}*"; return r[0]
    Q = lambda n: str(find_all(n)[0].parent)
    BUNDLE = W / "rfs_bundle"
    cmd = [str(CODE / "train" / "assemble.py")]
    p = subprocess.Popen([sys.executable, "-u", *cmd, "--out", str(BUNDLE), "--rfs_a", str(dsdir("rfs-a")), "--rfs_b", str(dsdir("rfs-b")), "--rfs_c", str(dsdir("rfs-c")),
                          "--wh_a", str(dsdir("whv3-a")), "--wh_b", str(dsdir("whv3-b")), "--wh_c", str(dsdir("whv3-c")), "--extmeta", Q("ext_v1_complete_face_meta.csv"),
                          "--rawv4", Q("train_English.txt"), "--gender_csv", str(CODE / "train" / "gender.csv")], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="", flush=True)
    assert p.wait() == 0, "assemble failed"
elif SMOKE:
    BUNDLE = W / "bundle_synth"; run(["synth", "--bundle", str(BUNDLE)]); LID = str(BUNDLE / "lid_strict.json")
    if MODE == "LAB":
        TABDIR = W / "lab_tables_synth"; run(["synthlab", "--bundle", str(BUNDLE), "--work", str(TABDIR)])
    torch.save({"params": [torch.zeros(2)]}, WORK / "lora_C1.pt")          # smoke: lora gia (luot L that chay tren Kaggle GPU)
elif True:
    cand = [p.parent for p in find_all("ext.npz") if (p.parent / "v4.npz").exists()]
    assert cand, "thieu bundle (ext.npz + v4.npz): gan dataset dac trung RFS-01 (FLAG_162 + adapter, README 'Hop dong bundle')"
    BUNDLE = cand[0]; l = sorted(find_all("lid_strict.json"), key=lambda q: "lid-strict" not in str(q));   # 13/11: the lid-strict dataset first (other inputs may hold a copy)
    assert l, "thieu lid_strict.json (ngnnguynchn/flag2027-lid-strict-p)"; LID = str(l[0])
    if MODE == "LAB":   # RFS-LAB-01: SAFE-04 pair tables of F1s (ngnnguynchn/flag2027-ct02s-r2ca-tables-p)
        t = find_all(os.environ.get("RFS_LAB_GLOB", "ct01r2_fold*.parquet").replace("*", "1")); assert t, "thieu bang fold SAFE-04 (flag2027-ct02s-r2ca-tables-p)"; TABDIR = t[0].parent
    if MODE in ("P", "LAB"):
        for n in ("lora_C1.pt", "v4_extra.npz"):
            f = find_all(n); assert f, f"thieu {n}: gan dataset dau ra cua luot L"; shutil.copy(f[0], WORK / n)
print("bundle", BUNDLE, "| lid", LID, flush=True)

In [ ]:
print("[cell] lora C1 (ImageBind-audio) + v4 lora/tuoi", flush=True)
if MODE == "L":
    import torch as _t
    ck = W / "imagebind_huge.pth"
    if not ck.exists():
        _t.hub.download_url_to_file("https://huggingface.co/PKU-Alignment/AnyRewardModel/resolve/40c7e2ade237848a194b2b77517da024c4ec8532/imagebind/imagebind_huge.pth", str(ck) + ".part", progress=False); os.replace(str(ck) + ".part", ck)
    tars = [p for p in find_all("tar0000.npz") if p.parent.name == "vox"]; assert tars, "thieu audio cache (duongphihoang/flag2027-audio-cache-p)"
    mav = None
    for d, dirs, files in os.walk(IN):
        if Path(d).name in ("v1", "v2") and {"faces", "voices"} <= set(dirs): mav = Path(d).parent; break
        dirs[:] = [x for x in dirs if x not in ("faces", "voices")] if len(Path(d).relative_to(IN).parts) < 7 else []
    assert mav, "thieu MAV raw (chnnhnnguyn/external-data-mavceleb)"
    v4 = None
    for d, dirs, files in os.walk(IN):
        if "train_English.txt" in files and Path(d, "faces").is_dir(): v4 = Path(d); break
        dirs[:] = [x for x in dirs if x not in ("faces", "voices")] if len(Path(d).relative_to(IN).parts) < 7 else []
    assert v4, "thieu raw-v4 (ngnnguynchn/flag2027-raw-v4-p)"
    run(["lora", "--bundle", str(BUNDLE), "--work", str(WORK), "--lid", LID, "--ib_ckpt", str(ck), "--cache", str(tars[0].parent.parent), "--mav", str(mav), "--v4root", str(v4)])
    OUTD = W / "rfs_lora"; OUTD.mkdir(exist_ok=True)
    for n in ("lora_C1.pt", "v4_extra.npz"): shutil.copy(WORK / n, OUTD / n)
else:
    print("mode P: lora tu dataset cua luot L", flush=True)

In [ ]:
print("[cell] FVBIND-01: train arms + SAFE-04 streams", flush=True)
p = CODE / "train" / "fvbind.py"; p.write_bytes(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/61cbW/bxrL+HiD/YS+Lg5KpTEuOk9MqUQDbsYvgJGmQ5KTBEQReSqIkRhSpkpRsV9f3t9/nmV2+Sk7b3KZFxJfd2dnZeZ9hLMu6+nT+6u3Lo25P2e8vz16+uVRpMA+zPEiDqeqdHvd6ahzMkjRQfnyr4s1qHKROX71a+fPgPIynyt9Mw0TlyXWQqh/U6+T9mbroAcgkXAf4OcpTP4wBK4lVskkx/CaMQj+9VesgzZI4e/jA/pTcXARRMD4BAFyfR2G8VMlM5YtAvb/6oNZJEnXUJAr82AtucnWsonDqZXkaTvJnBoGJP8HoSRDnQPWxyjrq/O3pyZFMctR1mC8ePlDqvKvUoI2k/fS0QKaDqTfqhNN76qTbVSDEGjd+PFkkKaZi3JEeWyPBzJ8ETkfg9wj/RHVPfyxA8lfAKPvn1J9eEE+nA3KAOn48TVZcTk0Wm3jZUU+rJTN/FZh1NegTgsYCPxTYDC+eZOp/1JX//vUIz3AUoEsc+KnaJuEkUIvAxxF2T07V0YtizjRcCbRXoMRAxYmmhZ0GMxx4jG24Dx+8A8J+uuqrAIc9nYbxPONpBNsAp7Y9NdDT5FoWLd+8OftUe8Wz+3B2dXnUPVW5P46CTG0y0Lqi2vUiiYKjWRgFKoyL5TucGPOvhw+yHPD9iJTCWQf+Sj3fxGFu+95MHanVxpthuDwJvK1+snVekNG4+CyJpsXK2P9sO36OTb3w+Px59sJd++lvmyBX9g/VO6yYB+4673DrKl7/TnK8DrcAEZPx8kSls8wTlnbXt8oOV+skzTMV5o6r3m9iNb5VV6/PfvZ6P/7UJ52VWt/mC6CENbBpTjo6Gm/iKXZ9jsvrJF2qX3GRbHL1C37B2eo1fg3mH3EZjr3Jcp2ri3/hRvP52QUuV/5WiH50tD1NkyRXn3DSR9hIBj7vnPcePviwSpZBX2P04c0v/7ocQMyz2xj0ycOJMnj8oK59bBGkDyHiQvWjMFarZBpEzjMFDeF9+Hj29uWrt5yOg4jU1M/9TmtoR82C65Lrj8kFmeNalvXwgSYTmGoOqmdBR33BGMgA2fxWFl7hvGcphGHt54soHCsz5Z1PwTU30D6gn4/DWJfP8iSdLDr6x41jeRs/fADuuc1cAnPDOAvS3O4S39QmQNvzyHae57hpkCXRNrAd8gOUh+MA8gyCAcb4g6HYolXnBstxgxtozsyGflTqO6iXdaIi/5ZHa0fJBHRLN3GGt5c3IBP2HOfH0G9QvscCRDPMtyGeDXsjYqThWbwUmJbsSNOqxJZUek8iCVNAESSZG8TbME1idx7ktlVxjOWowUBZPZyiYQIM19OgTvbmVbxSmxglc0x67+L34YOz928+4G5nnXetPvTRJLfPB09PO1oFZoPHHa0BBz3o3w5YM10NnnYLFTywwvHM0spW/7HOexWcEyjeElKvgPS0BPT4q4BOvg3Q5EkL0Ktqa3pee9U7kOHfn0EF29omN1ZH8edkLBdjy+HJPHzwHYT5//unJaIQfmoEqMjotg+jmEK3d4TZtWHPsPQk8rNMeW8+Gv119u+Xr34BqpbYWUvjZgZ9BD8v7Th23yTTTRQ4Zso0mCnPC6GZPc/OgmhWvBD23oD3wbblAKf2DmPB9muapth9LcbMxlHALJ9AE8lrP8/16zebKA9p487yHAIQJrH9+KSjMHjs55MFJCXN8gFQDIqpFMQ6ZA6nfXTqm/oIJfjq/O/eFMjvR2F+C/kjS5kdyAIvySc7Q95+QVTnTrAqVoYHBr6bysIdNa2vfQNg06GZP3JnUeLnNrYMBct1btxs4a+DYf9k9Exx/4fwKadXYBcYmvMw7BsqG8KwY/VIXKMTyAaOxcEqehgOxYYW5v/OsDtyV/C67F6NCGmQb9JYVdvMeRr2wqlgdwj6qOcUk+/0qXD32NLUEw62J8ti7xeiUyZwTQtqQ3Eb5VMjj7YMKz/e+JGXBcHU7gJtg485bMfNE/vCfXn56dXFJdT41o9s+BYQAg3HjL5wBZGQLgytOZGpkKSN9sbQ88vMDscd9eajsQNQtqJ1vShJ/T6USoChSQgLoh0n7cDKTDhF8H7m8BND6nRYTAqrRgIsgEFLOEsKuhYUT7HM3kG++eiKxI6gSzQy0CrDUZ1hjH0rgAAmxno0DpMlBl/5URbUx5c8eG0/gtX3ZgNMcQ1HdtSjR8v2aEGX1lwTf5OHUebW9m3sUfVkf7oheTXE9sD4XB9upJcGdPT9OB8IthoJbkwvGGYeaegFMb2oqe2oAKOAuWyAY5sL1vYDpppdP5MnhiR4QiXyNyplek+aZ8QX8+h+2cvgFqp4cPpjt9st6GmIEK9dHSq4mOND7XlpPLf9cWYv/GzBmY6j/qHsnnr+XD3uOY6rXed06sVJugIzx47rZ/ktxWytNQQUKoS56/YqBtZMOonCdWZfdhhjwWx11Mq/8aDdxDBDfAa9Aju4dojmKp8/U8sAbipc4DJU66jngwKA2obTINHxkPYSO9ggQhTNKfBq65GcDYt43FHb8bFD9914qCb0wo4z7VkKlWK6F4eJRIwh7ssgWINztcYokDObxOvxLU3O7q4StJBiBpBxEv8epIlNANRsTTG6HFpbL0sn1mgYjjgDVr1PaG4W5AYN26b31hjpaI9OP1sv9TNHxNT11+sAmiU0HMqdUmdDXMDHgLyv6ooxQ/4Ov4wE/y/EBgRwQbHVBoJO8xgFMZklc5xhnxL1Fe/Ne2dRxXcdZ2RsAswGFxlVNBJVRHg1oggbDEDWLCe2w+WIyhZ4ZIvNbBYFtkQFtMfQc7RFed1UlpTnqJZOKcgNLtLkpg7DUIHEQJS7442jnhcs15e3rj+d2o252iFo0xo8ltXYgCSmn1vDoxgR9uuSCycOxAohgyX+gG6gUPHUmWZ6U1ui2HO2GXNW5Q3CCWxtn8PWAlqzgo4HfFekxYG7DxiOO4+SsW09chG6wvE/oJX/oyWFdsxeOwe0NpZJO/poAziFQYqI2P49XNv/GVrCq4UiAQeDZfFUxPrA802eN58CIe5/aANVaOGlM6IHDLlMa4iMbyXEESI9U6swy2pc1+AR0FjbVxPGyvHSUX0OPF8cPxe8XljKhleHcD5CYH+r8wtGxYjc8HZMMyvDgV+TJkvtpI+tfXFndMMXYlrMye2Lde1ZyX1utobJtq1jzOg5w6PeqDZKyCYKYY//5VRAwX5BpbqaIWWXI6iohh6xw44yb3p7IIU3hcbF8LphxIIUKb521AuYihMYDC1kUZu1ENnZM0tVhNWGWwNHkKECd+6qHe+G/ccjnGztCPQ7URlA1aGnegeG3g2XlZqh3JAC2B8J//12/P1IBlot9QF2CvMAKZAwLqkkD+w2zk1ReIYszIBsC5G24CnH+i62RlwXvwT4H5fwMn3kb5GY2pdR0DsVhLlmXzSGMIv6dZgCYygmm6o3xp04LBiZiSojOA34V+Oxky3EGxgdMt7HiIn++fRHt2uMoFa8RjtwWSq5dEJBAh4za5dtVuCw74WDvy84OGvqXOfu+A/H8fjvLK23jM0z6kxzgc4wiBvRx7GCYTRkZgp3Bf/cEUwwAa4a81kCISa0SjztXWmv7oq0EvakRx5X8+FvZMgswwfZAeOSISj18DHL9dWLAXj4SY2HOyamNetNk0CfxCSBWy6INHC0RNmYJHmG5zCpqtfX+COYyaBAauhTp0zTBHI1bThzHA4RLRijQf5R5YzpDXs6bUBPJUY+GBM/17yvWjKaMRMJzHSo5KMNWjP1WdnF5aXJlyp/jiN6VubqmB1nbOyrGcPXQAOA15UsdTRQ+VomhF679IoyW1wKwQygP5d8i5saq0LY48asapJTeRPYG+KQG5KC5ga6cKZ1bUfpa2rU2v755wuhEpD4L+JoZUbjLh2nqcu+NASNDgPwCTO9Y/sGGjOKmLnDDulE/UBX8d/2zZDDR3T9ZBvmVc9Qg+aBm4N6fCYzkyWycPIMlx1Za9Q4fgLFQFYAlrXY8cnMvvSS1PvUcv0p8gt485OlPeRUuqTwXvPEg3HmnZ/5aerfmtkk1ORJRkLVDsCpZ6daf/4k0JmfRntQR9RCeJL9hjQlHhxQU39j1CR6BbakHqyMw7mJtREb+Km/AhddnXeMkK0lgtEllKKscq4TeB2pjgx4PggC0kEvOELiKPc3g67bfVJEOgh+xoOTJ09xj+xtwEu3WxPAsqCjXsWz5O3FZbtmQnUgdQ6dkyoljdEpBSxKMqlOyP21m7q5W4eAN5RxSUOyllbEqD8A3GTJS1fjcnVO7pZoKl+kyWa+UP9NGvy3sqmIoWGAzFrcIwxlfYZv6WX0WQSaJMirG3R/VD+fq9nYx4rCBAgrWQVEFeF8CCs8AvkQ6IUR4mqR2/dnb5xKP9STMs+QNdjivkisPPszkdp+ukZeaOhzbG0tDuHO0seNTFJx7laU4i7CUVvXQThf5N40mPi3eIYjvRuVTr2UxWq6oG/AFj5QDbT4JBzvyqMAVVH4EU6xFpjm8f5qZKU7g3CypuLTe8J1uHLPpv7qV1uvyMjTz+jrDudDgtTmYE4diNF6UU+PNfgfiFQ7Oi6ovHWcs7MXhdLpWB+OMddBshaPm5Ivcl8LN4HjOV6twtiG6FBz6+F8cVbuDQlYzLELWZtSDQz0K6MJOmQGJAoG+DHrHkiY4fxZl2Q9yOVfZKJsGdKI4jF07DTX8YBOZRrfh5n0yF+Np76ahMza9w1azCx4UjXSWm2SIE883yQbo9/A0iIHk3DEaSMGxkZpMyNIVLWW/E4xP9kTrwa/tRSoCR0XjFlyRJ6iZurpx6y5n6a3iuz7uLB47RPvCHPgJAveGGDsIzmJnouKgp1DD/SogqnQaoBFQhhz4zBAcXNgovpQkopg23XWjIY5rB/7kBG7ZpTRkABwHczJ8zz32kuo/iogBW/WFp8szOLl3C5Vcs2lPa+Npj4saneJ5Oz23HRWYgs2myCzPQzH9q7IcvbVZ3sSDqOkHyWgBiOU1VjOsv0M+fQqN1okyWUTGEX3y4/nAXEFlfQUp74tYIFM9W+bEOlqnVu0TWWBx0a3xmBfTRHdTm2IurZ9NtQ78FGB07LCA66JxEhbJBsLOYe0lA4N8FasFNYleLcwA/V15xRl4kuEoGLhPCxsJvvEhjX5b3/vWL51AD7AfYXmbXrfR+tnCMSBxDxojHQZbTgH9wEi0CVbyz72nLci0Wv8Nz3KOHE13hRNfm8uRb+GCdFrwHFr0cjocJdSXcetldWR/JhRU5V3KHnzvLQrNSWAZoU8a241Z5TV6/bVt/z5rt60wyVQaIY+slvBykkr/s9XzHbL8UAHSNllykJyv9c7lH7A8BfqMVAUHacksWtLkZZqgZHVY/avHGOgIwyqcwIlPogG81Xf7c0Q/B1LRwzBKXFzdgLzToO2DuYp/gQ8JJsLgEzY3g9VCP4PdfKEVO+yki301xs7QlR3OL0Bd1O38uxyBsoauMj5TsSRQT8UmHtKlAw/7MzFHZyWFgt0+253dgc9gMJjrnakPXJBsqNsVEc3r3GVCQxq1eWiYUivMzC/9GEnTIJ6fD84dMw45CLLgwdM8vSq1Mbf6bqvpLmBbjuvShXv05cpmkLcs3S+YUvEO96lttNKNNuWzvMxP8quGf7CleYPkvj80S0zvDIdM7yUYJwX6JaR1Kq0yiC9itWZGPZ8s6xtHR1Z0EpLqmXxmQZWmUvYH8o2G6s2VDpuLLA91SXGy544Iyurk7rotWZlJmNRcD095DPrCNtkdxkpyiN3tZyGMBq610Oq2mhSYruJlyyLInfe7R5yNqQjCE7TjRBTImzf5cMiD9mxhA9u4FWZrAnz8hzhUKR4w64NeI98poEiI4F2I93VAWuozwYY6wuMdHlIVAMSdbDhR0ZmReKalReXEQsMImoxhb6J4e0jhCUR41tZdngzQnlaPEtLZ13ZcFHbC/EsvXsaiXba3uCIZVLfFsThPRWhMniiCG9lvllf5ADOMjiCWdlwjKygU6T0q3Hfpq+hsMsurH7Vo6F94aaFfim8L0iDZ1/WAvyXVVwOauk3cE/LV+w0MVHD2ZV03PBJXzUC/HJgI7SXm2LMp9PDg5yqTlIS4+xqKKcjmX1JagAWf4F92c/wXdkYKGWeRkthBi8hikh1xqKNlsIyoK519mmAuK7lPg1/6QFgMamJtEtdaJPyXp+dez+//uVc1ETe7aUn0hz4qGgNtMoUErZ4H4CrX16//EB1YpAYriv3Qh5hrnjN9CFW1O8OX+347MapuLiCP2zBHtXEtAhjV6iFr7bady9KlFXGTISE+s6RckBzzPbAGA30I5mkwhNu0VQLqKEHPP8h1gCJSDEWsfm7TpM8mSQRr29FmkLRs6E1GrG5I8jREzMNbmwmQ7WmalLIsNGqmcEWAq22EnFsmWniHMkQf3TRnrEJMuPnbfls6m7DIZxd4KT+i3WZU2tUxODU+FVRUvu6NHnk663HTjtItrNXV9UpwlKbaPw6FTS5/+M66lsQ47HUUYu5f2VOPa/e6qbdDZm1WwVNYrK0wvaVSrQki65Xdu46IlLyaKWfWM691U7aa0lGlSk2TYIlzrPtUv+JQK7Z0GByms3A4sKF5ZacvGlDqFdZuSxrhQjSywi97Nk5FHG4k/UGf+v4n2yuq5tlst05WKH9K7vWuaiyO9N3tXfBmLzOXVLraa4txSx2QQ80txp7UwMF2SScy6+CaaJTtikJ/pqQVXo0uy8/W5SPOkV9+2AjiLgIDZusvepRyxLrndhDXYc0Rp05elPS1VnIKueg8/ff1C9RqyO06igSkjhlbnRQK9ZhF63ugv2Uk0xpFjlEWiCWzexPZRB7T1upnS8gZzMvJ0W5FgchA/VlJDnSFudfOyyUFszbPrnGamWB4x/qSZcuIEKaKlzSqdzdFwQstVrcfbFII+xgxyiRLmI5AQXNU6Rz7lCjw3U8vs0DdnyfPHqE8FACsp/PnT9eA7H/qpXNBAsVnFSj1GTGvLEwHt6MDmUJNYaDgUDY4a87tcO0Rn1YiwItQdVP6LsmWGAvI3Mumlee8ZuExLAsnM8GCxmC66R7mUZuZU3iiQdPpqNrRPdV8+AvAcnKvWX9XroTVxIbHeG8ATprlNnI2WaIlJuMMIldS5Y6odLiDlJZT4D0jaQS+sy0otScg5Wucr1rDB6S3JoAZktc9HB4LAzGEv6dLv9qq+DHZQ1GBuCmKuh2ah+m4NjiSZkovGvF60Ino3LaaljSOCJFDNPYZlmqPQjqIWERw140UH1rpkU3UBetDvIlDbMJSLKm6OydBLq7JINZxGCJIQ5ZxBkmaWshQQbE/kT3HOgHFAdEWqJVL6tRcI04cLQPcZ7kOgFYM2E114eeu8TBOug7sMqBqWY1x8TRBxqHJqS/baqJRegAVJohA3JErSHYf6uqKBzcO7BEwWaG7HZJcBitOpHZLa67GdBsi4ZDqY3vJplOfSA98wQKahWaZ2HxSIJ3Hf4Wz3XPwE8/MeVQO2vUx5Hun1r7KFJpDM1YD2t7ACPBkHZJCritzRX5tPv9wst3H+AT1kXgQH6uoUnKQPmAF2b0XNUIzyZ4kcSqgt/wrvaWopLlAvzQYR9+tpTEQNX0vFejHe7VZ1hdHNXMOEuNHfVOsKqKtwPZ4zkjUfkMQ255ZY2cw6fhbtZTGt+DqTISPFsebIv6OrSvV5+B+19C3RSj+dfesdYWFZ0qOBd61ik6YqvH1K51IN/Jh3Iwzvw6ri+ygEC6aHBh/25RV699uoUPGOFHNF5qhW4/Nt/EiGjFzFygwCUNvMiuBGlsvg2r1jcSRQtSM5YQKJSd9xfo8wsp+TgxT8D+C6CpbYgqPiMlwSrosjexTFq4u3+ttqR7Wg/Xo//p7PXOCsnRdQbitPSu1LnvrVEdZHY05WH5shjy9HQ0akndn61YVf1MT0+dthzw2KmZdXFp+7Xa0rYVMe2T9OtmWG8zY7BWq6EjTCVHlIUovUa7BsPPFYgWJki9Xb7E9CTJ/RWUhZXubW2BrQJ7ACJ/OnK31Xdb5oghWbjjD/M/szlu6DB2JAN7jI45fLIpjmT5WWjd7gWn/Eh0z9DqQNvY107rpUkhtE3ooZ6Eb4quF7OqEP5ZE9Ju1OebjT6fnbICWfJciwVgrzEHH6WkEnuU5bbPutKGUIh1FvhuOiXtSHSkVzmgj0uaLWZ2wOCJv/XKNfMCYJ/f7b0jYD2TTc0FhQdlAGlSG4B8OuACq+brVfF2NQjQ9UNWGGh+IB8M5JvhCoFPp/waUtiOJfniNAP51OnNp8ZLc5qBFLuusPIVX+979Kx5H/LzWy3gHZ3q/XhPt+qKyXb7pU5xmaQB1YXRKGyMROvDlvX8l+hSrb0pYkfGDVgEmbJ7XraKpL6ndaM+/dVp2RqBnZ4Oay86snSXGa6ry+Fq1nzX5Wt8RdACDw4x+cvZhpzEz9RrglDJuOSqeEoFYzXcxSbQra9BmuQSTlOcKJ1D7Miaok1+06BxoOV7eLjbUTlGEosdjDO9GVtscLWvXLNJ4fSSWuabdOPjAhfzSXrh0R5Iv6C7ZTL837pkbX0jWo6J7IBQqx4tR8yWLSkXDSspEU8zm+AhT7jI2u7JEjO4u+yuSnPvV8dtZHzW7OPXvUiudKSMka3ZT/uODmkiRpbzBmYNVl3qrdW6Lw+4W0iQ6WCWdclu7ROP3J67tzV4/MhEPjY+6R+2BWUqhIUD7Byl1v5T/qzT/k/ZHVrgdGQ6R1xqOmB3yzJaeIxo4fLyvdq9dwPUJdlLD7rsYdB3T2aNmJXZDUlXFEpDmxpUZQcpe5jbDQHdqmhe5TCmlIM/nawJx0UD3WQz9ZEGXOPDQKmA2rWv4+QdPpHztwhjmEwuvo9rJMkQGNJ1N5Fh/5t6Esy/6tAXX1OqlfT0in9+A+rV1HXwOXJtJ6sZ1XhUsr8cBCtLUoMSfdottYG8bFWrDObtmlbX+btVblsUye8XPezbuk9471PSw9Xp6KuKuj2gpRNQ4ZwMqW6bKIhuqpNuOEPP+RFJfCAWv5d2w62eFd0Tnd+nbwSR+/RNfaBHYcE/CuPtvve+d7/ge09dfr5zWTgG1a9TnJGuFEspeQqqoDyAWUyITVEUZ7duUY2Xr43qtRP9r9xoaaJgZYcyoIVUoa/EFKHaFRYW4oFHylK8o2sn/C/kl+Ec6XnCJ57HNgfPK2Jj3fTw8MH/ATgb+FFvRwAA")))
STANDIN = os.environ.get("FVB_STANDIN") == "1"
FOUT = W / "fvb_out"; FOUT.mkdir(exist_ok=True)
ck, cache, mav, v4 = "", "", "", ""
if not SMOKE:
    if not STANDIN:
        for c in ("pip -q install iopath ftfy regex fvcore", "pip -q install --no-deps git+https://github.com/facebookresearch/ImageBind"):
            print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout[-300:], flush=True)
        ck = W / "imagebind_huge.pth"
        if not ck.exists():
            torch.hub.download_url_to_file("https://huggingface.co/PKU-Alignment/AnyRewardModel/resolve/40c7e2ade237848a194b2b77517da024c4ec8532/imagebind/imagebind_huge.pth", str(ck) + ".part", progress=False); os.replace(str(ck) + ".part", ck)
        import hashlib
        h = hashlib.sha256()
        with open(ck, "rb") as f:
            for b in iter(lambda: f.read(1 << 24), b""): h.update(b)
        assert h.hexdigest().startswith("d6f6c22b"), "ImageBind checkpoint SHA-256 mismatch"; print("imagebind sha256 ok", flush=True)
    tars = [q for q in find_all("tar0000.npz") if q.parent.name == "vox"]; assert tars, "thieu audio cache (duongphihoang/flag2027-audio-cache-p)"
    cache = tars[0].parent.parent; assert (cache / "vb").is_dir(), "audio cache without vb/"
    for d, dirs, files in os.walk(IN):
        if Path(d).name in ("v1", "v2") and {"faces", "voices"} <= set(dirs): mav = Path(d).parent; break
        dirs[:] = [x for x in dirs if x not in ("faces", "voices")] if len(Path(d).relative_to(IN).parts) < 7 else []
    assert mav, "thieu MAV raw (chnnhnnguyn/external-data-mavceleb)"
    for d, dirs, files in os.walk(IN):
        if "train_English.txt" in files and Path(d, "faces").is_dir(): v4 = Path(d); break
        dirs[:] = [x for x in dirs if x not in ("faces", "voices")] if len(Path(d).relative_to(IN).parts) < 7 else []
    assert v4, "thieu raw-v4 (ngnnguynchn/flag2027-raw-v4-p)"
t0 = time.time()
r = subprocess.Popen([sys.executable, "-u", str(p), "--bundle", str(BUNDLE), "--work", str(WORK), "--out", str(FOUT), "--lid", LID,
                      "--tables", str(TABDIR), "--ib_ckpt", str(ck), "--cache", str(cache), "--mav", str(mav), "--v4root", str(v4),
                      "--arms", os.environ.get("FVB_ARMS", "B0,B1")], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in r.stdout: print(line, end="", flush=True)
assert r.wait() == 0, "fvbind.py failed"
print(f"FVBIND done [{time.time() - t0:.0f}s]:", sorted(q.name for q in FOUT.iterdir()), flush=True)

In [ ]:
print("[cell] upload", flush=True)
UPL = os.environ.get("RFS_UPLOAD", "")
OUT = {"L": W / "rfs_lora", "B": W / "rfs_bundle"}.get(MODE, ART)
if SMOKE:
    print("smoke: no upload; artifact", sorted(p.name for p in OUT.iterdir()) if OUT.exists() else None, flush=True)
elif UPL:
    try:
        import kagglehub
        kagglehub.dataset_upload(UPL, str(OUT), version_notes=f"RFS-01 FLAG_163 mode {MODE}")
        print("uploaded ->", UPL, flush=True)
    except Exception as e:
        print(f"upload to {UPL} failed ({e!r}); files are in the notebook output", flush=True)
print("ALL CELLS OK", flush=True)